## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../../src").resolve()))
from condpred import report
from condpred.experiments import e1

SMOKE = os.environ.get("CONDPRED_SMOKE") == "1"
OUT = Path("../../notebooks/tables/condpred/E1" + ("/smoke" if SMOKE else ""))
FIG = OUT / "figures"
CACHE = "E1/smoke" if SMOKE else "E1"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

SPEC = e1.build_initial_spec(SMOKE)
TRAJ = {}
HORIZONS = {}
EMB = {}

## 2. Write spec

In [ ]:
report.write_spec(SPEC, OUT, modules=e1.E1_MODULES, completion_keys=frozenset({"horizons"}))
e1.verify_spec_immutable(OUT, SPEC, e1.E1_MODULES)

## 3. Leakage test

In [ ]:
rng = np.random.default_rng(0)
x = rng.standard_normal(600)
leaks = e1.run_leakage_e1(x, rng)
ldf = pd.DataFrame(leaks, columns=["pipeline", "pass"])
report.save_table(ldf, OUT, "leakage")
assert ldf["pass"].all(), ldf[~ldf["pass"]]

## 4. Generate trajectories

In [ ]:
TRAJ = e1.load_trajectories(SPEC, CACHE)

## 5. Choose (m, tau); horizons; complete spec

In [ ]:
SPEC, emb_df, HORIZONS, EMB = e1.run_embedding_and_horizons(SPEC, TRAJ, OUT)
report.save_table(emb_df, OUT, "embedding_choice")

## 6. Forecasts and losses

## 7. Features

## 8. Fit predictors; primary metrics

## 9. Primary decision

In [ ]:
RESULT = e1.run_primary_and_outputs(SPEC, OUT, FIG, TRAJ, HORIZONS, EMB)

## 10. Secondary analyses

## 11. Figures and tables

In [ ]:
decision = pd.read_csv(OUT / "decision.csv")
display(decision)